In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [13]:
city_df_clean = pd.read_csv("data/city_safety/city_safety_2023_clean.csv")
city_df_clean.head()

,Rank,city,country,crime_index,safety_index,year,city_norm,country_norm
0,1,Caracas,Venezuela,83.6,16.4,2023,caracas,venezuela
1,2,Pretoria,South Africa,82.0,18.0,2023,pretoria,south africa
2,3,Durban,South Africa,81.0,19.0,2023,durban,south africa
3,4,Port Moresby,Papua New Guinea,80.7,19.3,2023,port moresby,papua new guinea
4,5,Johannesburg,South Africa,80.7,19.3,2023,johannesburg,south africa


In [14]:
city_df_geo = pd.read_csv("data/city_safety/city_safety_2023_geo.csv")
city_df_geo.head()

,Rank,city,country,crime_index,safety_index,year,city_norm,country_norm,lat,lon
0,1,Caracas,Venezuela,83.6,16.4,2023,caracas,venezuela,10.506093,-66.914601
1,2,Pretoria,South Africa,82.0,18.0,2023,pretoria,south africa,-25.745928,28.187910
2,3,Durban,South Africa,81.0,19.0,2023,durban,south africa,-29.861825,31.009909
3,4,Port Moresby,Papua New Guinea,80.7,19.3,2023,port moresby,papua new guinea,-9.474330,147.159950
4,5,Johannesburg,South Africa,80.7,19.3,2023,johannesburg,south africa,-26.205000,28.049722


In [15]:
country_df = pd.read_csv("data/crime-rate-by-country-2024.csv")

In [16]:
# Standardize column names from the raw ta_data table
country_df = country_df.rename(columns={
    "Country": "country",
    "crimeRateByCountry_crimeIndex": "crime_index",
    "CrimeRate_OverallCriminalityScoreGOCI": "overall_criminality_score",
    "CrimeRate_CriminalMarketsScore": "criminal_markets_score",
    "CrimeRate_CriminalActorsScore": "criminal_actors_score",
    "CrimeRate_ResilienceScore": "resilience_score",
    "CrimeRateSafetyIndex": "safety_index",
})

country_df.columns.tolist()

['country',
 'crime_index',
 'overall_criminality_score',
 'criminal_markets_score',
 'criminal_actors_score',
 'resilience_score',
 'safety_index']

In [17]:
country_df.head()

,country,crime_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index
0,India,44.4,5.75,6.70,4.8,5.42,55.6
1,China,60.8,6.37,6.53,6.2,5.67,39.2
2,United States,49.2,5.67,5.83,5.5,7.13,50.8
3,Indonesia,45.9,6.85,6.60,7.1,4.25,54.1
4,Pakistan,42.8,6.03,6.27,5.8,3.96,57.2


In [18]:
city_df_geo['country'] = city_df_geo['country'].str.strip().str.lower()
city_df_geo['country_norm'] = city_df_geo['country_norm'].str.strip().str.lower()
country_df['country'] = country_df['country'].str.strip().str.lower()
country_df['country_norm'] = country_df['country'].str.strip().str.lower()

In [19]:
country_df.head()

,country,crime_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index,country_norm
0,india,44.4,5.75,6.70,4.8,5.42,55.6,india
1,china,60.8,6.37,6.53,6.2,5.67,39.2,china
2,united states,49.2,5.67,5.83,5.5,7.13,50.8,united states
3,indonesia,45.9,6.85,6.60,7.1,4.25,54.1,indonesia
4,pakistan,42.8,6.03,6.27,5.8,3.96,57.2,pakistan


In [20]:
city_df_geo.head()

,Rank,city,country,crime_index,safety_index,year,city_norm,country_norm,lat,lon
0,1,Caracas,venezuela,83.6,16.4,2023,caracas,venezuela,10.506093,-66.914601
1,2,Pretoria,south africa,82.0,18.0,2023,pretoria,south africa,-25.745928,28.187910
2,3,Durban,south africa,81.0,19.0,2023,durban,south africa,-29.861825,31.009909
3,4,Port Moresby,papua new guinea,80.7,19.3,2023,port moresby,papua new guinea,-9.474330,147.159950
4,5,Johannesburg,south africa,80.7,19.3,2023,johannesburg,south africa,-26.205000,28.049722


In [21]:
# Map state/province codes to actual countries
country_fix = {
    # United States states / DC
    'md': 'united states',
    'tn': 'united states',
    'mi': 'united states',
    'nm': 'united states',
    'mo': 'united states',
    'la': 'united states',
    'ca': 'united states',
    'wi': 'united states',
    'il': 'united states',
    'pa': 'united states',
    'oh': 'united states',
    'tx': 'united states',
    'ga': 'united states',
    'ak': 'united states',
    'dc': 'united states',
    'in': 'united states',
    'ky': 'united states',
    'fl': 'united states',
    'mn': 'united states',
    'nv': 'united states',
    'or': 'united states',
    'az': 'united states',
    'wa': 'united states',
    'ny': 'united states',
    'hi': 'united states',
    'nc': 'united states',
    'co': 'united states',
    'ma': 'united states',
    'id': 'united states',
    'ut': 'united states',

    # Canadian provinces
    'ab': 'canada',
    'bc': 'canada',
}

# Apply mapping on the city dataframe
city_df_geo['country_norm'] = (
    city_df_geo['country_norm']
    .str.strip()
    .str.lower()
    .replace(country_fix)
)

In [22]:
# Country columns we want to merge
country_feats = [
    'country_norm',
    'crime_index',
    'overall_criminality_score',
    'criminal_markets_score',
    'criminal_actors_score',
    'resilience_score',
    'safety_index'
]

city_master = city_df_geo.merge(
    country_df[country_feats],
    on='country_norm',
    how='left',
    suffixes=('', '_country')
)

In [24]:
# Check how many cities failed to find a country match
print(city_master['crime_index_country'].isna().mean())

# Check head
city_master.head()

0.11298076923076923


,Rank,city,country,crime_index,safety_index,year,city_norm,country_norm,lat,lon,crime_index_country,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index_country
0,1,Caracas,venezuela,83.6,16.4,2023,caracas,venezuela,10.506093,-66.914601,82.1,6.72,6.03,7.4,1.88,17.9
1,2,Pretoria,south africa,82.0,18.0,2023,pretoria,south africa,-25.745928,28.187910,75.5,7.18,6.87,7.5,5.63,24.5
2,3,Durban,south africa,81.0,19.0,2023,durban,south africa,-29.861825,31.009909,75.5,7.18,6.87,7.5,5.63,24.5
3,4,Port Moresby,papua new guinea,80.7,19.3,2023,port moresby,papua new guinea,-9.474330,147.159950,80.4,5.72,5.33,6.1,3.29,19.6
4,5,Johannesburg,south africa,80.7,19.3,2023,johannesburg,south africa,-26.205000,28.049722,75.5,7.18,6.87,7.5,5.63,24.5


In [25]:
city_master[city_master['crime_index_country'].isna()][
    ['city', 'country', 'country_norm']
].head(20)

,city,country,country_norm
28,Cali,colombia,colombia
38,Santo Domingo,dominican republic,dominican republic
43,Bogota,colombia,colombia
58,Lethbridge,ab,canada
60,Surrey,canada,canada
69,Kelowna,canada,canada
71,Red Deer,canada,canada
80,Sudbury,canada,canada
89,Sault Ste. Marie,canada,canada
91,Winnipeg,canada,canada


In [26]:
missing_countries = (
    city_master.loc[city_master['crime_index_country'].isna(), 'country_norm']
    .value_counts()
)
print(missing_countries)

canada                37
colombia               3
thailand               3
dominican republic     1
moldova                1
georgia                1
hong kong (china)      1
Name: country_norm, dtype: int64


In [27]:
# Make sure country_df has a normalized name column
country_df['country_norm'] = (
    country_df['country'].str.strip().str.lower()
)

check_list = [
    'canada',
    'colombia',
    'thailand',
    'dominican republic',
    'moldova',
    'georgia',
    'hong kong (china)',
]

for name in check_list:
    print('---', name, '---')
    print(
        country_df[country_df['country_norm'].str.contains(name.split()[0])]
        [['country', 'country_norm']]
        .head(5)
    )


--- canada ---
   country country_norm
37  canada       canada
--- colombia ---
     country country_norm
27  colombia     colombia
--- thailand ---
     country country_norm
19  thailand     thailand
--- dominican republic ---
               country        country_norm
84  dominican republic  dominican republic
--- moldova ---
     country country_norm
137  moldova      moldova
--- georgia ---
     country country_norm
130  georgia      georgia
--- hong kong (china) ---
       country country_norm
104  hong kong    hong kong


short_codes = (
    city_master.loc[
        city_master['crime_index_country'].isna(), 'country_norm'
    ]
    .dropna()
    .unique()
)

# Filter to 2-letter strings
short_codes = [c for c in short_codes if len(c) == 2]
print(short_codes)